In [ ]:
!pip install -q faiss-cpu pypdf openai tiktoken numpy

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

In [ ]:
from pypdf import PdfReader

def read_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

raw_text = read_pdf(pdf_path)
print("Total characters:", len(raw_text))

In [ ]:
def chunk_text(text, chunk_size=800, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap

    return chunks

chunks = chunk_text(raw_text)
print("Total chunks:", len(chunks))

In [ ]:
import openai
import numpy as np

client = openai.OpenAI()

def embed_texts(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    vectors = [e.embedding for e in response.data]
    return np.array(vectors).astype("float32")

embeddings = embed_texts(chunks)

In [ ]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index ready")

In [ ]:
def retrieve(query, top_k=3):
    query_embedding = embed_texts([query])
    distances, indices = index.search(query_embedding, top_k)

    retrieved_chunks = [chunks[i] for i in indices[0]]
    return retrieved_chunks


In [ ]:
def generate_answer(query, context_chunks):
    context = "\n\n".join(context_chunks)

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [ ]:
def ask_question(query):
    context = retrieve(query)
    answer = generate_answer(query, context)

    print("Question:", query)
    print("\n Answer:\n", answer)

# Example
ask_question("What is the main topic of the document?")